# **Part A: Conceptual Foundation**

### **1. What is Regularization in Machine Learning? Why is it needed?**

Regularization is a machine learning technique used to reduce model complexity by adding a penalty term to the loss function during training. It helps control overfitting, where a model learns training data too well but fails on unseen data. Regularization improves generalization and makes the model more stable. Common regularization methods include Ridge (L2) and Lasso (L1). It is especially useful when working with datasets having many features.

---

### **2. Difference between Ridge Regression (L2) and Lasso Regression (L1)**

**Ridge Regression (L2):**
- Adds squared coefficient penalty to the cost function.
- Reduces feature weights but keeps all features in the model.
- Useful when most input features contribute to prediction.

**Lasso Regression (L1):**
- Adds absolute coefficient penalty to the cost function.
- Can reduce some coefficients to zero.
- Useful for automatic feature selection and simpler models.

---

### **3. What is Cross-Validation and why is it important?**
Cross-validation is a validation technique used to check how well a machine learning model performs on different parts of data. It helps in selecting a stable model, reducing overfitting, and giving more reliable performance evaluation before final deployment.

---

### **4. Explain the following cross-validation techniques:**

*  **K-Fold Cross-Validation**
The dataset is divided into multiple equal parts called folds. The model is trained using some folds and tested on the remaining fold. This repeats until every fold is used for testing once, and the average result is taken.

*  **Stratified K-Fold Cross-Validation**
This method works like K-Fold but keeps the target distribution balanced in each fold. It provides fair evaluation when the dataset distribution is uneven.

*  **Leave-One-Out Cross-Validation (LOOCV)**
In this method, one sample is used for testing and all remaining samples are used for training. The process repeats until every sample has been tested once. It gives detailed evaluation but takes more computation time.

*  **Time Series Split**
Time Series Split is used when data has a time sequence. Training is done on earlier data and testing is done on later data. This prevents future information from affecting model training.

---

### **5. Why are tree-based models less sensitive to feature scaling?**
Tree-based models like Decision Tree and Random Forest split data based on conditions and threshold values instead of distance calculations. Feature scaling changes the range of values but does not change their order. Because of this, model decisions remain mostly unaffected. Unlike algorithms such as SVR or Linear Regression, scaling is not necessary for tree-based methods. This makes preprocessing simpler.

# **Part B: Dataset Understanding & Preparation**

In [65]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV, LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold, StratifiedKFold, LeaveOneOut, TimeSeriesSplit, cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [4]:
df = pd.read_csv("Advanced_Regression_HousePrice_Dataset.csv")
df.head()

,property_id,sale_date,area_sqft,bedrooms,bathrooms,location_score,property_age,distance_city_km,near_school,near_metro,crime_rate_index,house_price_inr
0,200001,2014-01-01,2181,6,4,8.1,21,3.8,0,0,4.84,35154898
1,200002,2019-12-01,2383,5,4,5.3,28,10.9,1,1,2.89,26710893
2,200003,2016-10-01,1047,3,3,5.9,7,27.5,0,1,4.04,11216242
3,200004,2013-03-01,1753,4,3,7.0,27,12.1,0,0,3.28,21984310
4,200005,2013-07-01,1728,4,4,10.0,32,1.4,0,1,3.84,25080429


In [5]:
print("Shape         :", df.shape)
print("Columns name  :",df.columns)

Shape         : (3800, 12)
Columns name  : Index(['property_id', 'sale_date', 'area_sqft', 'bedrooms', 'bathrooms',
       'location_score', 'property_age', 'distance_city_km', 'near_school',
       'near_metro', 'crime_rate_index', 'house_price_inr'],
      dtype='object')


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3800 entries, 0 to 3799
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   property_id       3800 non-null   int64  
 1   sale_date         3800 non-null   object 
 2   area_sqft         3800 non-null   int64  
 3   bedrooms          3800 non-null   int64  
 4   bathrooms         3800 non-null   int64  
 5   location_score    3800 non-null   float64
 6   property_age      3800 non-null   int64  
 7   distance_city_km  3800 non-null   float64
 8   near_school       3800 non-null   int64  
 9   near_metro        3800 non-null   int64  
 10  crime_rate_index  3800 non-null   float64
 11  house_price_inr   3800 non-null   int64  
dtypes: float64(3), int64(8), object(1)
memory usage: 356.4+ KB


In [7]:
df.describe()

,property_id,area_sqft,bedrooms,bathrooms,location_score,property_age,distance_city_km,near_school,near_metro,crime_rate_index,house_price_inr
count,3800.00000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3800.000000,3.800000e+03
mean,201900.50000,1716.925526,3.428158,2.916316,6.502237,22.537105,13.085132,0.548421,0.472895,4.242911,2.071940e+07
std,1097.10984,582.996559,1.356682,1.133540,1.766945,12.325740,6.537425,0.497715,0.499330,2.045371,8.707465e+06
min,200001.00000,500.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.500000,1.506126e+06
25%,200950.75000,1322.000000,2.000000,2.000000,5.300000,14.000000,8.500000,0.000000,0.000000,2.810000,1.446895e+07
50%,201900.50000,1700.500000,3.000000,3.000000,6.500000,20.000000,13.000000,1.000000,0.000000,4.220000,1.989180e+07
75%,202850.25000,2105.000000,4.000000,4.000000,7.700000,29.000000,17.500000,1.000000,1.000000,5.650000,2.596062e+07
max,203800.00000,3776.000000,7.000000,6.000000,10.000000,80.000000,38.700000,1.000000,1.000000,12.000000,5.930315e+07


### 6. Identify features and target variable

In [8]:
features = ['area_sqft', 'bedrooms', 'bathrooms', 'location_score','property_age', 'distance_city_km', 'near_school','near_metro', 'crime_rate_index']
target   = 'house_price_inr'

print("Features :", features)
print("Target   :", target)

Features : ['area_sqft', 'bedrooms', 'bathrooms', 'location_score', 'property_age', 'distance_city_km', 'near_school', 'near_metro', 'crime_rate_index']
Target   : house_price_inr


In [9]:
X = df[features]
y = df[target]

### 7. Train-Test Split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

print("X_train Shape:", X_train.shape)
print("X_test Shape:", X_test.shape)
print("y_train Shape:", y_train.shape)
print("y_test Shape:", y_test.shape)

X_train Shape: (3040, 9)
X_test Shape: (760, 9)
y_train Shape: (3040,)
y_test Shape: (760,)


### 8: Basic Preprocessing (Scaling)

In [11]:
scal = StandardScaler()
X_train_scal = scal.fit_transform(X_train)
X_test_scal = scal.transform(X_test)

# **Part C: Regularized Linear Models**

In [12]:
results = []

def evaluate_model(model, model_name, X_train, X_test, y_train, y_test):

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"{model_name} Results:")
    print()
    print(f"MSE: {mse:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE: {mae:.2f}")
    print(f"R2 Score: {r2:.2f}")

    results.append([model_name, mse, rmse, mae, r2])

    return model

### 9. Ridge Regression (L2)

In [13]:
ridge = Ridge(alpha=1.0)

l2 = evaluate_model(ridge,"Ridge Regression (L2)", X_train_scal, X_test_scal, y_train, y_test)

Ridge Regression (L2) Results:

MSE: 6545419443619.51
RMSE: 2558401.74
MAE: 1959198.97
R2 Score: 0.92


### 10. Lasso Regression (L1)

In [14]:
lasso = Lasso(alpha=1.0)

l1 = evaluate_model(lasso,"Lasso Regression (L1)", X_train_scal, X_test_scal, y_train, y_test)

Lasso Regression (L1) Results:

MSE: 6543996619352.28
RMSE: 2558123.65
MAE: 1959378.45
R2 Score: 0.92


### 11. Tune alpha using Cross Validation

In [15]:
ridge_cv = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100], cv=5)

ridge_cv.fit(X_train_scal, y_train)

print("Best Ridge Alpha:", ridge_cv.alpha_)

Best Ridge Alpha: 1.0


In [16]:
ridge_best = Ridge(alpha=ridge_cv.alpha_)

evaluate_model( ridge_best, "Ridge CV", X_train_scal, X_test_scal, y_train, y_test)

Ridge CV Results:

MSE: 6545419443619.51
RMSE: 2558401.74
MAE: 1959198.97
R2 Score: 0.92


Ridge(alpha=np.float64(1.0))

In [17]:
lasso_cv = LassoCV(alphas=[0.01, 0.1, 1, 10, 100], cv=5, max_iter=10000)

lasso_cv.fit(X_train_scal, y_train)

print("Best Lasso Alpha:", lasso_cv.alpha_)

Best Lasso Alpha: 100.0


In [18]:
lasso_best = Lasso(alpha=lasso_cv.alpha_)

evaluate_model(lasso_best, "Lasso CV", X_train_scal, X_test_scal, y_train, y_test)

Lasso CV Results:

MSE: 6544107497269.08
RMSE: 2558145.32
MAE: 1959385.12
R2 Score: 0.92


Lasso(alpha=np.float64(100.0))

In [19]:
results_df = pd.DataFrame(
    results,
    columns=["Model", "MSE", "RMSE", "MAE", "R2 Score"])

results_df

,Model,MSE,RMSE,MAE,R2 Score
0,Ridge Regression (L2),6.545419e+12,2.558402e+06,1.959199e+06,0.918726
1,Lasso Regression (L1),6.543997e+12,2.558124e+06,1.959378e+06,0.918744
2,Ridge CV,6.545419e+12,2.558402e+06,1.959199e+06,0.918726
3,Lasso CV,6.544107e+12,2.558145e+06,1.959385e+06,0.918742


In [20]:
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Ridge Coefficient": ridge_best.coef_,
    "Lasso Coefficient": lasso_best.coef_
})

coef_df

,Feature,Ridge Coefficient,Lasso Coefficient
0,area_sqft,6.937545e+06,6.944831e+06
1,bedrooms,2.973655e+05,2.911937e+05
2,bathrooms,2.853268e+05,2.853815e+05
3,location_score,3.676865e+06,3.678911e+06
4,property_age,-6.513024e+05,-6.513909e+05
5,distance_city_km,-3.016962e+04,-2.873756e+04
6,near_school,1.697147e+04,1.680345e+04
7,near_metro,5.820432e+04,5.819679e+04
8,crime_rate_index,-1.354728e+05,-1.353088e+05


# **Part D: Cross-Validation Strategies**

### 13. K-Fold Cross Validation

In [42]:
ridge = Ridge(alpha=1.0)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

kf_scores = cross_val_score( ridge, X_train_scal, y_train, cv=kf, scoring='r2')

print("K-Fold R2 Scores        :", kf_scores)
print("Average K-Fold R2 Score :", np.mean(kf_scores))

K-Fold R2 Scores        : [0.91723295 0.90705947 0.91475732 0.91968897 0.91739335]
Average K-Fold R2 Score : 0.9152264116132749


In [44]:
y_bins = pd.qcut(y_train, q=5, labels=False)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

skf_scores = cross_val_score(ridge, X_train_scal, y_train, cv=skf.split(X_train_scal, y_bins), scoring='r2')

print("Stratified K-Fold R2 Scores :", skf_scores)
print("Average Stratified R2 Score :", np.mean(skf_scores))

Stratified K-Fold R2 Scores : [0.91098549 0.92207826 0.91657936 0.91704764 0.9110434 ]
Average Stratified R2 Score : 0.9155468309171642


In [60]:
loo = LeaveOneOut()

loo_scores = cross_val_score( ridge, X_train_scal[:200], y_train.iloc[:200], cv=loo, scoring='neg_mean_squared_error')

print("LOOCV R2 Score      :", loo_scores)
print("Average LOOCV Score :", np.mean(loo_scores))

LOOCV R2 Score      : [-1.37850249e+12 -2.07342956e+13 -5.67473478e+11 -9.63809123e+10
 -8.94249748e+12 -6.68418720e+12 -4.91745449e+11 -2.58714045e+11
 -3.18863541e+11 -2.09839685e+13 -5.87174981e+10 -8.82587502e+11
 -7.96660822e+12 -3.97921918e+13 -3.48762052e+13 -1.88603078e+12
 -2.36206389e+12 -1.60352119e+11 -1.07712506e+12 -2.55102676e+11
 -1.89035653e+13 -7.82170974e+11 -3.39436921e+12 -6.31518190e+12
 -1.33372021e+12 -5.45649303e+09 -7.45879334e+10 -2.58361452e+12
 -3.28876445e+12 -2.67113673e+13 -6.78421778e+11 -4.18998167e+13
 -8.61786028e+11 -1.22457285e+12 -3.36284762e+11 -5.33125204e+10
 -4.59631223e+10 -5.81010913e+11 -1.60514150e+12 -6.48159807e+10
 -5.99657030e+11 -1.34023416e+10 -9.10866249e+12 -1.09700649e+12
 -3.58467249e+12 -5.24273126e+13 -3.61941424e+11 -1.89624063e+13
 -3.97812483e+11 -4.35220889e+12 -1.07808068e+12 -1.09506854e+13
 -6.47966136e+12 -1.63783147e+12 -9.91198509e+12 -3.82682670e+12
 -6.02108952e+12 -4.68777772e+10 -1.37356120e+11 -3.03222011e+12
 -4

In [51]:
tscv = TimeSeriesSplit(n_splits=5)

ts_scores = cross_val_score(ridge, X_train_scal, y_train, cv=tscv, scoring='r2')

print("Time Series Scores        :", ts_scores)
print("Average Time Series Score :", np.mean(ts_scores))

Time Series Scores        : [0.90652599 0.91433262 0.92198442 0.9167497  0.91429032]
Average Time Series Score : 0.9147766121364629


In [56]:
cv_results = pd.DataFrame({
    "CV Method": [
        "K-Fold",
        "Stratified K-Fold",
        "LOOCV",
        "Time Series Split"],

    "Average R2 Score": [
        np.mean(kf_scores),
        np.mean(skf_scores),
        np.mean(loo_scores),
        np.mean(ts_scores)]})

cv_results

,CV Method,Average R2 Score
0,K-Fold,9.152264e-01
1,Stratified K-Fold,9.155468e-01
2,LOOCV,-5.858074e+12
3,Time Series Split,9.147766e-01


# **Part E: Tree-Based Regression Models**

In [70]:
dt = DecisionTreeRegressor(random_state=42)

dt_model = evaluate_model(dt, "Decision Tree", X_train, X_test, y_train, y_test)

Decision Tree Results:

MSE: 11418620582615.76
RMSE: 3379144.95
MAE: 2480762.05
R2 Score: 0.86


In [67]:
dt_tuned = DecisionTreeRegressor(
    max_depth=5,
    min_samples_split=10,
    random_state=42
)

dt_tuned_model = evaluate_model(dt_tuned, "Decision Tree Tuned", X_train, X_test, y_train, y_test)

Decision Tree Tuned Results:

MSE: 9385836459040.17
RMSE: 3063631.25
MAE: 2325303.89
R2 Score: 0.88


In [71]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model = evaluate_model(rf, "Random Forest", X_train, X_test, y_train, y_test)

Random Forest Results:

MSE: 5842748489016.42
RMSE: 2417177.79
MAE: 1778261.34
R2 Score: 0.93


In [69]:
results_df = pd.DataFrame(
    results,
    columns=["Model", "MSE", "RMSE", "MAE", "R2 Score"]
)

results_df

,Model,MSE,RMSE,MAE,R2 Score
0,Ridge Regression (L2),6.545419e+12,2.558402e+06,1.959199e+06,0.918726
1,Lasso Regression (L1),6.543997e+12,2.558124e+06,1.959378e+06,0.918744
2,Ridge CV,6.545419e+12,2.558402e+06,1.959199e+06,0.918726
3,Lasso CV,6.544107e+12,2.558145e+06,1.959385e+06,0.918742
4,Decision Tree,1.141862e+13,3.379145e+06,2.480762e+06,0.858216
5,Decision Tree Tuned,9.385836e+12,3.063631e+06,2.325304e+06,0.883457
6,Random Forest,5.842748e+12,2.417178e+06,1.778261e+06,0.927451


# **Part F: Support Vector Regression**

# **Part G: Model Comparison & Evaluation**

# **Part H: Final Analysis & Reporting**